# LangGraph Agent 动手实践
## 2026-05-11 | 第2周周一 | StateGraph + Checkpoint 持久化

> **学习目标**: 用 LangGraph 把前三天手写的 while 循环重构为图结构，
> 并通过 SQLite Checkpoint 实现跨会话对话记忆。

**你将学到：**
- `StateGraph` 的核心概念：节点、边、条件路由
- `add_messages` Reducer — 为什么 LangGraph 的 State 不需要手动 append
- `ToolNode` + `tools_condition` — 工具调用的图结构实现
- `SqliteSaver` — 把对话历史持久化到磁盘，实现跨会话记忆
- DeepSeek 思维模型的 `reasoning_content` 兼容处理

**与前三天的关系：**
```
w1d1 手写循环:  while loop → if tool_calls → execute → append
w1d2 MCP 改造:  同上，但工具执行通过标准化协议
w1d3 弹性封装:  同上，加重试/熔断/降级
今天 LangGraph: StateGraph → chatbot node → [tools?] → chatbot → END
```

---

## 新手导读：LangGraph 把 while loop 变成可视化状态机

如果 Week 1 的 Agent 是手写 `while True`，LangGraph 的价值是把循环拆成“节点”和“边”。

核心词汇：

- State：图在每一步携带的数据，本例里主要是 `messages`。
- Node：一次可执行步骤，例如调用模型的 `chatbot` 节点，或执行工具的 `tools` 节点。
- Edge：节点之间怎么走；固定边是确定流转，条件边根据模型是否有 tool calls 分叉。
- Checkpoint：把 state 落盘，方便恢复对话和调试。

阅读顺序建议：

1. 先看 `State`，它决定图里流动的“数据形状”。
2. 再看工具定义，LangGraph 的 ToolNode 会依赖这些工具。
3. 看 `chatbot` 节点，理解模型调用仍然是普通 API 调用。
4. 看 `add_conditional_edges`，这是“有工具调用就去 tools，否则结束”的关键。

常见卡点：

- LangGraph 不是替你思考，它只是让控制流更明确、更可恢复。
- StateGraph 不是流程图图片，它是可以执行的状态机。
- `add_messages` 的作用是合并消息，而不是每次覆盖整个历史。


## 1. 环境准备

LangGraph 依赖链：
- `langgraph` — 图引擎本体
- `langgraph-checkpoint-sqlite` — SQLite 持久化后端
- `langchain-core` — 消息类型（AIMessage, HumanMessage…）
- `openai` — 直接调用 DeepSeek API（绕过 langchain-openai 的 reasoning_content 问题）

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `1. 环境准备`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [1]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

import json
import math
import os
import sys
from typing import Annotated

import openai as openai_module
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# 验证 API 连接
raw_client = openai_module.OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.deepseek.com",
)
models = raw_client.models.list()
for m in models.data:
    print(f"  {m.id}")
print("连接成功！使用模型: deepseek-v4-flash")

c:\Users\tallm\Documents\Codes\agent-building\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


  deepseek-v4-flash
  deepseek-v4-pro
连接成功！使用模型: deepseek-v4-flash


## 2. 工具定义 — `@tool` 装饰器 vs 手写 JSON Schema

前三天我们手写了完整的 JSON Schema：
```python
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "...",
        "parameters": { "type": "object", "properties": {...} }
    }
}
```

LangChain 的 `@tool` 装饰器自动从函数签名和 docstring 生成等价的 Schema：

| 项目 | 来源 |
|------|------|
| `name` | 函数名 |
| `description` | docstring 第一行 |
| `parameters` | 函数签名的类型注解 |
| `required` | 无默认值的参数 |

好处：描述和代码在同一处，避免「Schema 说一套，代码做另一套」的同步问题。

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `2. 工具定义 — `@tool` 装饰器 vs 手写 JSON Schema`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [2]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """查询指定城市的实时天气信息。返回温度、天气状况、湿度、风速。

    Args:
        city: 城市名称，支持中文或英文，例如：北京、Tokyo、London
        unit: 温度单位，celsius（摄氏度）或 fahrenheit（华氏度），默认 celsius
    """
    weather_db = {
        "北京":    {"temp_c": 22, "condition": "晴",     "humidity": 40, "wind": "北风 3级"},
        "上海":    {"temp_c": 25, "condition": "多云",   "humidity": 68, "wind": "东南风 2级"},
        "广州":    {"temp_c": 29, "condition": "雷阵雨", "humidity": 85, "wind": "南风 4级"},
        "深圳":    {"temp_c": 28, "condition": "阴",     "humidity": 78, "wind": "东风 3级"},
        "杭州":    {"temp_c": 24, "condition": "小雨",   "humidity": 72, "wind": "东北风 2级"},
        "成都":    {"temp_c": 21, "condition": "阴",     "humidity": 75, "wind": "无持续风向 1级"},
        "tokyo":   {"temp_c": 18, "condition": "晴",     "humidity": 50, "wind": "北风 2级"},
        "london":  {"temp_c": 13, "condition": "小雨",   "humidity": 80, "wind": "西风 5级"},
        "new york":{"temp_c": 16, "condition": "多云",   "humidity": 55, "wind": "西南风 4级"},
        "sydney":  {"temp_c": 20, "condition": "晴",     "humidity": 45, "wind": "东风 3级"},
        "paris":   {"temp_c": 15, "condition": "阴",     "humidity": 70, "wind": "西南风 3级"},
    }
    key = city.strip().lower()
    data = weather_db.get(key, {"temp_c": 20, "condition": "暂无数据", "humidity": 60, "wind": "未知"})
    temp = data["temp_c"]
    unit_label = "°C"
    if unit == "fahrenheit":
        temp = round(temp * 9 / 5 + 32, 1)
        unit_label = "°F"
    return json.dumps({
        "city": city, "temperature": temp, "unit": unit_label,
        "condition": data["condition"], "humidity": f"{data['humidity']}%",
        "wind": data["wind"],
    }, ensure_ascii=False)


@tool
def calculate(expression: str) -> str:
    """安全执行数学表达式计算。

    支持：四则运算(+ - * /)、幂运算(**)、三角函数(sin/cos/tan)、
    平方根(sqrt)、对数(log/log10)、绝对值(abs)。

    Args:
        expression: 数学表达式字符串，如 '(2+3)*4'、'sqrt(144)'、'2**10'
    """
    allowed = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    allowed.update({"abs": abs, "round": round, "min": min, "max": max, "pow": pow})
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)
        return json.dumps({"expression": expression, "result": result, "error": None}, ensure_ascii=False)
    except Exception as e:
        return json.dumps({"expression": expression, "result": None, "error": str(e)}, ensure_ascii=False)


TOOLS = [get_weather, calculate]

# @tool 自动生成的 Schema，可以直接传给 OpenAI API
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description,
            "parameters": t.args_schema.model_json_schema(),
        },
    }
    for t in TOOLS
]

# 查看自动生成的 Schema
print("@tool 自动生成的 get_weather Schema:")
print(json.dumps(TOOL_SCHEMAS[0], indent=2, ensure_ascii=False))

@tool 自动生成的 get_weather Schema:
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "查询指定城市的实时天气信息。返回温度、天气状况、湿度、风速。\n\nArgs:\n    city: 城市名称，支持中文或英文，例如：北京、Tokyo、London\n    unit: 温度单位，celsius（摄氏度）或 fahrenheit（华氏度），默认 celsius",
    "parameters": {
      "description": "查询指定城市的实时天气信息。返回温度、天气状况、湿度、风速。\n\nArgs:\n    city: 城市名称，支持中文或英文，例如：北京、Tokyo、London\n    unit: 温度单位，celsius（摄氏度）或 fahrenheit（华氏度），默认 celsius",
      "properties": {
        "city": {
          "title": "City",
          "type": "string"
        },
        "unit": {
          "default": "celsius",
          "title": "Unit",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "title": "get_weather",
      "type": "object"
    }
  }
}


## 3. State 定义 — 图的「记忆单元」

### LangGraph State 的核心思想

LangGraph 中，所有节点共享一个 `State` 对象。每个节点执行后返回的是 **增量更新**，不是完整替换。

```
前三天手写：                      LangGraph：
messages = [...]  # 局部变量      state["messages"]  # 持久化对象
messages.append(new_msg)         return {"messages": [new_msg]}  # 自动追加
```

### `add_messages` Reducer

```python
class State(TypedDict):
    messages: Annotated[list, add_messages]
```

`Annotated[list, add_messages]` 告诉 LangGraph：当两个节点都往 `messages` 写入时，
不是后者覆盖前者，而是**追加**（append）。这正是 `add_messages` Reducer 的作用。

### 【八股题 24】StateGraph vs 手写循环

| 维度 | 手写循环 | StateGraph |
|------|---------|------------|
| **消息历史** | 函数内局部变量，调用结束即消失 | State 持久化，跨调用存活 |
| **控制流** | `if/while` 代码逻辑 | 显式图结构，可视化 |
| **并发** | 需要手动处理 | 并行节点开箱即用 |
| **回放/调试** | 无内建支持 | Checkpoint 可回放任意历史状态 |

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `3. State 定义 — 图的「记忆单元」`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [3]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

class State(TypedDict):
    messages: Annotated[list, add_messages]

SYSTEM_PROMPT = """你是一个具备工具调用能力的智能助手。你拥有以下工具：

1. get_weather — 查询任意城市的实时天气（温度、天气状况、湿度、风速）
2. calculate   — 执行数学表达式计算（支持四则运算、幂运算、三角函数等）

行为准则：
- 用户询问天气相关信息时，主动调用 get_weather
- 用户需要数值计算时，调用 calculate，禁止自行心算
- 收到工具返回结果后，用流畅的中文向用户转述
- 保持回答简洁、信息密度高"""

print("State 定义完成")
print("messages 字段使用 add_messages Reducer — 节点返回的新消息会自动追加，而非覆盖")

State 定义完成
messages 字段使用 add_messages Reducer — 节点返回的新消息会自动追加，而非覆盖


## 4. 消息格式转换 — DeepSeek reasoning_content 兼容

### 问题背景

DeepSeek 的思维模型（deepseek-v4-flash）在 API 响应中会附带 `reasoning_content` 字段（模型的内部推理过程）。
当这条消息被再次传回 API 时，**必须原样带上 `reasoning_content`**，否则 DeepSeek 会返回 400 错误。

### 为什么不直接用 `langchain-openai`？

`langchain-openai` 的 `ChatOpenAI` 在将 `AIMessage` 序列化为 OpenAI 格式时，
不会自动传递 `reasoning_content`（非标准字段），导致第二轮请求失败。

### 解决方案

在 `chatbot` 节点内直接使用 `openai` SDK 发起 API 调用，手动做格式转换：
- `_to_openai_format()` — LangChain 消息 → OpenAI API 格式，保留 `reasoning_content`
- `_from_openai_response()` — OpenAI 响应 → LangChain `AIMessage`，把 `reasoning_content` 存入 `additional_kwargs`

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `4. 消息格式转换 — DeepSeek reasoning_content 兼容`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [4]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

def _to_openai_format(messages: list) -> list:
    """将 LangChain 消息列表转换为 OpenAI API 格式，保留 reasoning_content。"""
    result = []
    for msg in messages:
        if msg.type == "system":
            result.append({"role": "system", "content": msg.content})
        elif msg.type == "human":
            result.append({"role": "user", "content": msg.content or ""})
        elif msg.type == "ai":
            d: dict = {"role": "assistant", "content": msg.content or ""}
            rc = (msg.additional_kwargs or {}).get("reasoning_content")
            if rc:
                d["reasoning_content"] = rc
            if msg.tool_calls:
                d["content"] = None
                d["tool_calls"] = [
                    {
                        "id": tc["id"],
                        "type": "function",
                        "function": {
                            "name": tc["name"],
                            "arguments": json.dumps(tc["args"]),
                        },
                    }
                    for tc in msg.tool_calls
                ]
            result.append(d)
        elif msg.type == "tool":
            result.append({
                "role": "tool",
                "tool_call_id": msg.tool_call_id,
                "content": msg.content,
            })
    return result


def _from_openai_response(msg) -> AIMessage:
    """将 OpenAI 响应消息转为 LangChain AIMessage，保留 reasoning_content。"""
    kwargs: dict = {}
    rc = getattr(msg, "reasoning_content", None)
    if rc:
        kwargs["additional_kwargs"] = {"reasoning_content": rc}

    if msg.tool_calls:
        tool_calls = [
            {
                "id": tc.id,
                "name": tc.function.name,
                "args": json.loads(tc.function.arguments),
                "type": "tool_call",
            }
            for tc in msg.tool_calls
        ]
        return AIMessage(content=msg.content or "", tool_calls=tool_calls, **kwargs)

    return AIMessage(content=msg.content or "", **kwargs)


print("消息格式转换函数已定义")

消息格式转换函数已定义


## 5. 图构建 — StateGraph 的核心结构

### 图拓扑

```
START ──→ chatbot ──[tools_condition: 有 tool_calls]──→ tools ──┐
                 └──[tools_condition: 无 tool_calls]──→ END      │
          ↑                                                      │
          └────────────────────────────────────────────────────-─┘
```

两个节点：
1. `chatbot` — 调用 LLM，决定是否需要工具
2. `tools` — 执行工具，把结果追加到 State

### 条件边 vs 固定边

```python
# 固定边：总是从 tools 回到 chatbot
graph.add_edge("tools", "chatbot")

# 条件边：chatbot 的出口取决于最后一条消息是否有 tool_calls
graph.add_conditional_edges("chatbot", tools_condition)
# tools_condition 内部逻辑：
#   if last_message.tool_calls → "tools"
#   else                       → END
```

### `ToolNode` — 工具调用的标准化封装

`ToolNode(TOOLS)` 自动：
1. 读取 State 最后一条消息的 `tool_calls`
2. 找到对应的 `@tool` 函数并执行
3. 将每个调用的结果包装为 `ToolMessage` 追加到 State

等同于前三天手写的：
```python
for tc in msg.tool_calls:
    result = TOOL_EXECUTORS[tc.function.name](**json.loads(tc.function.arguments))
    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
```

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `5. 图构建 — StateGraph 的核心结构`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [5]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

def build_graph(checkpointer=None):
    """构建并编译 LangGraph Agent 图。"""

    def chatbot(state: State) -> dict:
        """LLM 节点：注入 System Prompt，调用 DeepSeek API。"""
        messages = state["messages"]
        if not messages or messages[0].type != "system":
            messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(messages)

        openai_msgs = _to_openai_format(messages)
        response = raw_client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=openai_msgs,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.0,
        )
        return {"messages": [_from_openai_response(response.choices[0].message)]}

    tool_node = ToolNode(TOOLS)

    graph = StateGraph(State)
    graph.add_node("chatbot", chatbot)
    graph.add_node("tools", tool_node)

    graph.add_edge(START, "chatbot")
    graph.add_conditional_edges("chatbot", tools_condition)
    graph.add_edge("tools", "chatbot")

    return graph.compile(checkpointer=checkpointer)


# 先用无状态模式（不传 checkpointer）测试基础功能
app_stateless = build_graph()
print("图编译成功！")
print()
print("图结构（节点 + 边）:")
print("  节点: chatbot, tools")
print("  边:   START→chatbot, chatbot→[条件]→tools/END, tools→chatbot")

图编译成功！

图结构（节点 + 边）:
  节点: chatbot, tools
  边:   START→chatbot, chatbot→[条件]→tools/END, tools→chatbot


### 5.1 可视化图结构

`get_graph().draw_mermaid()` 输出 Mermaid 语法的图描述，可以直接在 Jupyter 中渲染。

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `5.1 可视化图结构`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [6]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

from IPython.display import HTML, display as ipy_display

# 编译一个无状态图用于查看拓扑
_app_viz = build_graph()

mermaid_code = _app_viz.get_graph().draw_mermaid()

print("Mermaid 源代码:")
print(mermaid_code)

# 在 Jupyter 中渲染 Mermaid 图（依赖 cdn.jsdelivr.net，需联网）
ipy_display(HTML(f"""
<pre style="background:#1e1e2e;padding:16px;border-radius:8px;color:#cdd6f4;overflow-x:auto;">{mermaid_code}</pre>
"""))

# 如果想用 PNG 渲染（需要额外依赖）:
#   pip install pygraphviz           # Python 包
#   winget install graphviz          # 系统级 graphviz 二进制
# 然后:
# from IPython.display import Image
# display(Image(_app_viz.get_graph(xray=True).draw_mermaid_png()))

Mermaid 源代码:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chatbot(chatbot)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> chatbot;
	chatbot -.-> __end__;
	chatbot -.-> tools;
	tools --> chatbot;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 6. 基础测试 — 验证图功能

`.invoke()` 的调用方式：
```python
result = app.invoke(
    {"messages": [HumanMessage(content="...")]},  # 初始输入
    config={"configurable": {"thread_id": "..."}},  # 会话 ID（Checkpoint 用）
)
# 返回最终 State，result["messages"][-1].content 是最终回复
```

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `6. 基础测试 — 验证图功能`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [7]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

def chat(app, user_input: str, thread_id: str = "default", verbose: bool = True) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
    )
    final = result["messages"][-1].content
    if verbose:
        print(f"  用户: {user_input}")
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  → 工具调用: {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")
            elif msg.type == "tool":
                preview = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
                print(f"  ← 工具返回: {preview}")
        print(f"  Agent: {final}\n")
    return final


# 测试 1: 单工具
print("=" * 50)
print("测试 1: 单工具 — 天气查询")
print("=" * 50)
chat(app_stateless, "北京今天天气怎么样？", thread_id="nb-1")

测试 1: 单工具 — 天气查询
  用户: 北京今天天气怎么样？
  → 工具调用: get_weather({"city": "北京", "unit": "celsius"})
  ← 工具返回: {"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"...
  Agent: 北京今天天气不错！☀️

- **天气状况**：晴
- **温度**：22°C
- **湿度**：40%
- **风速**：北风 3级

很适合出门活动的一天～



'北京今天天气不错！☀️\n\n- **天气状况**：晴\n- **温度**：22°C\n- **湿度**：40%\n- **风速**：北风 3级\n\n很适合出门活动的一天～'

In [8]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

# 测试 2: 并行调用
print("=" * 50)
print("测试 2: 并行调用 — 同时查两个城市")
print("=" * 50)
chat(app_stateless, "上海和广州的天气分别怎么样？", thread_id="nb-2")

测试 2: 并行调用 — 同时查两个城市
  用户: 上海和广州的天气分别怎么样？
  → 工具调用: get_weather({"city": "上海"})
  → 工具调用: get_weather({"city": "广州"})
  ← 工具返回: {"city": "上海", "temperature": 25, "unit": "°C", "condition": "多云", "humidity": "68%", "wind": "东南风 2...
  ← 工具返回: {"city": "广州", "temperature": 29, "unit": "°C", "condition": "雷阵雨", "humidity": "85%", "wind": "南风 4...
  Agent: **上海** 🌤 多云，气温 **25°C**，湿度 68%，东南风 2 级，体感舒适。

**广州** ⛈ 雷阵雨，气温 **29°C**，湿度 85%，南风 4 级，比较闷热，出门记得带伞！



'**上海** 🌤 多云，气温 **25°C**，湿度 68%，东南风 2 级，体感舒适。\n\n**广州** ⛈ 雷阵雨，气温 **29°C**，湿度 85%，南风 4 级，比较闷热，出门记得带伞！'

In [9]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

# 测试 3: 多步推理（查天气 + 计算）
print("=" * 50)
print("测试 3: 多步推理 — 查天气再计算")
print("=" * 50)
chat(app_stateless, "北京比成都热几度？如果成都再降 3 度呢？", thread_id="nb-3")

测试 3: 多步推理 — 查天气再计算
  用户: 北京比成都热几度？如果成都再降 3 度呢？
  → 工具调用: get_weather({"city": "北京", "unit": "celsius"})
  → 工具调用: get_weather({"city": "成都", "unit": "celsius"})
  ← 工具返回: {"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"...
  ← 工具返回: {"city": "成都", "temperature": 21, "unit": "°C", "condition": "阴", "humidity": "75%", "wind": "无持续风向 ...
  → 工具调用: calculate({"expression": "22 - 21"})
  → 工具调用: calculate({"expression": "22 - (21 - 3)"})
  ← 工具返回: {"expression": "22 - 21", "result": 1, "error": null}
  ← 工具返回: {"expression": "22 - (21 - 3)", "result": 4, "error": null}
  Agent: 当前天气情况如下：

- **北京** ☀️ 22°C，晴，湿度 40%
- **成都** ☁️ 21°C，阴，湿度 75%

### 温差计算
1. **现在**：北京比成都 **热 1°C**（22 - 21 = 1）
2. **如果成都再降 3°C**（变为 18°C）：北京比成都 **热 4°C**（22 - 18 = 4）

北京是干热（湿度低），成都湿冷感会更明显一些~



'当前天气情况如下：\n\n- **北京** ☀️ 22°C，晴，湿度 40%\n- **成都** ☁️ 21°C，阴，湿度 75%\n\n### 温差计算\n1. **现在**：北京比成都 **热 1°C**（22 - 21 = 1）\n2. **如果成都再降 3°C**（变为 18°C）：北京比成都 **热 4°C**（22 - 18 = 4）\n\n北京是干热（湿度低），成都湿冷感会更明显一些~'

## 7. Checkpoint 持久化 — 跨会话记忆

### 【八股题 27】Checkpoint 的核心价值

```
传统 Agent（手写循环）:
  第 1 次调用: run_agent("北京天气")  → messages = [sys, user, ai, tool, ai]
  第 2 次调用: run_agent("刚才多少度?") → messages = [sys, user, ai]  ← 重新开始！

LangGraph + Checkpoint:
  第 1 次调用: app.invoke({...}, config={thread_id: "t1"})
    → State 写入 SQLite: {thread_id: "t1", messages: [...]}
  第 2 次调用: app.invoke({...}, config={thread_id: "t1"})
    → 从 SQLite 读取 t1 的历史，追加新消息，继续对话
```

### `thread_id` 的隔离语义

| thread_id | 效果 |
|-----------|------|
| 相同 | 共享消息历史，实现多轮记忆 |
| 不同 | 完全隔离，互不干扰（多用户独立会话） |

### SQLite Checkpoint 使用方式

```python
# 作为 context manager 使用（自动关闭连接）
with SqliteSaver.from_conn_string("checkpoints.db") as checkpointer:
    app = build_graph(checkpointer=checkpointer)
    app.invoke(...)  # State 自动持久化到 SQLite
```

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `7. Checkpoint 持久化 — 跨会话记忆`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [10]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

DB_PATH = "checkpoints_nb.db"

with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
    app = build_graph(checkpointer=checkpointer)

    print("=" * 50)
    print("多轮对话测试 — 同一 thread_id 保留历史")
    print("=" * 50)

    print("--- 第 1 轮: 查天气 ---")
    chat(app, "北京今天天气怎么样？", thread_id="multi-turn")

    print("--- 第 2 轮: 引用上一轮数据 ---")
    chat(app, "刚才北京的温度换算成华氏度是多少？", thread_id="multi-turn")

    print("--- 第 3 轮: 继续引用 ---")
    chat(app, "北京的风力适合放风筝吗？", thread_id="multi-turn")

多轮对话测试 — 同一 thread_id 保留历史
--- 第 1 轮: 查天气 ---
  用户: 北京今天天气怎么样？
  → 工具调用: get_weather({"city": "北京"})
  ← 工具返回: {"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"...
  → 工具调用: calculate({"expression": "22 * 9/5 + 32"})
  ← 工具返回: {"expression": "22 * 9/5 + 32", "result": 71.6, "error": null}
  → 工具调用: get_weather({"city": "北京"})
  ← 工具返回: {"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"...
  Agent: 北京今天天气情况如下：

- **温度**：22°C（舒适宜人）
- **天气状况**：☀️ 晴
- **湿度**：40%（比较干爽）
- **风力**：北风 3级

和刚才的数据一致，依然是晴好舒适的一天，户外活动完全没问题～

--- 第 2 轮: 引用上一轮数据 ---
  用户: 刚才北京的温度换算成华氏度是多少？
  → 工具调用: get_weather({"city": "北京"})
  ← 工具返回: {"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"...
  → 工具调用: calculate({"expression": "22 * 9/5 + 32"})
  ← 工具返回: {"expression": "22 * 9/5 + 32", "result": 71.6, "error": null}
  → 工具调用: get_weather({"city": "北京"})
  ← 工具返回: {"city": 

In [11]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

# 会话隔离测试
with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
    app = build_graph(checkpointer=checkpointer)

    print("=" * 50)
    print("会话隔离测试 — 不同 thread_id 互不干扰")
    print("=" * 50)

    print("--- 全新会话（thread_id='isolated'）---")
    chat(app, "刚才我们聊到北京了吗？", thread_id="isolated")

会话隔离测试 — 不同 thread_id 互不干扰
--- 全新会话（thread_id='isolated'）---
  用户: 刚才我们聊到北京了吗？
  Agent: 没有，这是我们对话的**第一条消息**，之前没有聊到任何城市哦。😄

如果你想查询北京的天气，我现在就可以帮你查！



## 8. 深入：查看 Checkpoint 存储的 State

`get_state()` 可以查看任意 thread 当前的完整 State，
`get_state_history()` 可以查看该 thread 的全部历史快照（支持时间旅行调试）。

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `8. 深入：查看 Checkpoint 存储的 State`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [12]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
    app = build_graph(checkpointer=checkpointer)

    config = {"configurable": {"thread_id": "multi-turn"}}
    state = app.get_state(config)

    print(f"thread_id='multi-turn' 的当前 State")
    print(f"共 {len(state.values['messages'])} 条消息：\n")

    for i, msg in enumerate(state.values["messages"]):
        role = msg.type
        if role == "ai" and hasattr(msg, "tool_calls") and msg.tool_calls:
            names = [tc['name'] for tc in msg.tool_calls]
            print(f"  [{i:2d}] assistant → tool_calls: {names}")
        elif role == "tool":
            preview = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
            print(f"  [{i:2d}] tool        → {preview}")
        else:
            content_preview = (msg.content or "")[:80].replace("\n", " ")
            print(f"  [{i:2d}] {role:10s} → {content_preview}")

thread_id='multi-turn' 的当前 State
共 20 条消息：

  [ 0] human      → 北京今天天气怎么样？
  [ 1] assistant → tool_calls: ['get_weather']
  [ 2] tool        → {"city": "北京", "temperature": 22, "unit": "°C", "condition":...
  [ 3] ai         → 北京今天天气如下：  - **温度**：22°C - **天气状况**：☀️ 晴 - **湿度**：40% - **风速**：北风 3级  整体来说是晴朗舒适的
  [ 4] human      → 刚才北京的温度换算成华氏度是多少？
  [ 5] assistant → tool_calls: ['calculate']
  [ 6] tool        → {"expression": "22 * 9/5 + 32", "result": 71.6, "error": nul...
  [ 7] ai         → 北京当前的 22°C 换算成华氏度是 **71.6°F**，相当于温暖的春日温度，体感舒适。
  [ 8] human      → 北京的风力适合放风筝吗？
  [ 9] ai         → 根据刚才查到的北京天气数据，当前是 **北风3级**，非常适合放风筝！🎏  - **3级风**（风速约 3.4~5.4 m/s）被称为"微风"，能让风筝稳定升空
  [10] human      → 北京今天天气怎么样？
  [11] assistant → tool_calls: ['get_weather']
  [12] tool        → {"city": "北京", "temperature": 22, "unit": "°C", "condition":...
  [13] ai         → 北京今天天气情况如下：  - **温度**：22°C（舒适宜人） - **天气状况**：☀️ 晴 - **湿度**：40%（比较干爽） - **风力**：北风 
  [14] human      → 刚才北京的温度换算成华氏度是多少？
  [15] assistant → to

## 9. 自由探索区

试试这些扩展：
- 改 `thread_id` 开始全新对话，或继续已有对话
- 问一个需要 3+ 轮工具调用的问题（如「分别查北京、上海、广州天气，哪个最适合出门」）
- 用 `get_state_history()` 查看某个 thread 的所有历史快照

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `9. 自由探索区`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


In [13]:
# 学习注释：这一段属于「这一块怎么读」，主题是 LangGraph StateGraph。
# 阅读顺序：先看输入/输出变量，再看核心函数调用，最后看结果如何交给下一步。
# 本段重点：State、Node、Edge、ToolNode、checkpoint 和条件边。
# 新手提醒：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。

with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
    app = build_graph(checkpointer=checkpointer)

    # ===== 修改这里：thread_id 和 user_input =====
    user_input = "深圳今天天气怎么样？帮我算一下 1024 * 768"
    thread_id  = "free-explore"
    # ============================================

    chat(app, user_input, thread_id=thread_id)

  用户: 深圳今天天气怎么样？帮我算一下 1024 * 768
  → 工具调用: get_weather({"city": "深圳"})
  → 工具调用: calculate({"expression": "1024 * 768"})
  ← 工具返回: {"city": "深圳", "temperature": 28, "unit": "°C", "condition": "阴", "humidity": "78%", "wind": "东风 3级"...
  ← 工具返回: {"expression": "1024 * 768", "result": 786432, "error": null}
  Agent: 刚才已经查过啦，再告诉你一遍 😊

**深圳今日天气：**
- 🌡️ 温度：28°C
- ☁️ 天气状况：阴
- 💧 湿度：78%
- 🌬️ 风力：东风 3级

**计算：1024 × 768 = 786,432** ✅

天气湿度偏高，体感可能有点闷，注意补水哦~



## 10. 知识点回顾

### 今天你掌握了什么

| # | 知识点 | 对应 Cell |
|---|--------|-----------|
| 1 | **`@tool` 装饰器** — 自动从函数签名生成 Tool Schema，替代手写 JSON | Cell 2 |
| 2 | **`State` + `add_messages`** — Reducer 机制，增量更新而非覆盖 | Cell 3 |
| 3 | **`StateGraph`** — 节点/边/条件路由，显式图结构替代隐式 while 循环 | Cell 5 |
| 4 | **`ToolNode` + `tools_condition`** — 工具调用的图结构标准实现 | Cell 5 |
| 5 | **`SqliteSaver`** — Checkpoint 持久化，thread_id 隔离会话 | Cell 7-8 |
| 6 | **reasoning_content 兼容** — DeepSeek 思维模型的多轮对话处理 | Cell 4 |

### 对应八股题

| 题号 | 主题 | 对应 Cell |
|------|------|-----------|
| 题 24 | StateGraph vs 手写循环 | Cell 3 |
| 题 27 | Checkpoint 持久化原理 | Cell 7 |

---

> **下一步**: Multi-Agent 协作（`multi-agent-collab/`）
> 多个 LangGraph 图通过消息传递协作完成复杂任务。

### 这一块怎么读

这段对应 **LangGraph StateGraph** 里的 `10. 知识点回顾`。新手阅读时不要先背 API 名字，先抓住三件事：

1. 这一块接收什么输入，产出什么输出。
2. 它在完整 Agent 流程里处在哪个位置。
3. 它和上一块之间靠什么数据结构连接。

本节重点：State、Node、Edge、ToolNode、checkpoint 和条件边。

容易误解：LangGraph 不是魔法，它只是把手写循环变成可恢复的图。


## 学习检查清单

读完这节后，建议你能回答：

- State / Node / Edge 各自对应手写 Agent 里的哪一部分？
- `tools_condition` 根据什么决定去 tools 还是 END？
- 为什么 checkpoint 可以实现跨轮对话恢复？
- DeepSeek 的 `reasoning_content` 为什么需要特殊处理？
- 什么时候应该用 LangGraph，而不是继续手写 while loop？
